# Part B — Predictive modeling, continuing from the same cleaned data

In [38]:
!pip3 install scikit-learn --break-system-packages
!pip install imbalanced-learn --break-system-packages

In [39]:
from sklearn.model_selection import train_test_split
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler




# Section 9.Split the data into train/test

In [40]:
titanic_dataframe = pd.read_csv('titanic.csv')

# Features
X = titanic_dataframe.drop(columns=['survived'])

# Target
y = titanic_dataframe['survived']

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [42]:
print("Original Distribution")
print(y.value_counts(normalize=True) * 100)

print("\nTraining Distribution")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Distribution")
print(y_test.value_counts(normalize=True) * 100)

Original Distribution
survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64

Training Distribution
survived
0    61.657303
1    38.342697
Name: proportion, dtype: float64

Testing Distribution
survived
0    61.452514
1    38.547486
Name: proportion, dtype: float64


### The dataset was split into training and testing sets using a stratified train-test split, with survived as the target variable. 

#### Stratification was used because the target classes are not perfectly balanced (approximately 62% did not survive and 38% survived in the Titanic dataset). 
#### A random split without stratification could produce training or testing sets with noticeably different class proportions, leading to biased model training or unreliable evaluation. 
#### By using stratify=y, both subsets preserve the original class distribution, provides more reliable model evaluation.

#### The training set might contain relatively more survivors than the test set (or vice versa).

#### This imbalance can bias the model and distort evaluation metrics such as accuracy, precision, and recall.

# Section 8. Preprocessing : Everything must be fit only on the training data.

### List of columns to be considered

In [43]:
X = titanic_dataframe[
    [
        'pclass',
        'sex',
        'age',
        'sibsp',
        'parch',
        'fare',
        'embarked'
    ]
]

y = titanic_dataframe['survived']

### Split the dataset into 80-20 ratio

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### Numeric and Categorical features

In [45]:
#Numeric Features

numeric_features = [
    'age',
    'fare',
    'sibsp',
    'parch',
    'pclass'
]

#Categorical Features
categorical_features = [
    'sex',
    'embarked'
]

### Numeric Pipeline

In [46]:
# Numeric Pipeline
numeric_pipeline = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='median')
        ),

        (
            'scaler',
            StandardScaler()
        )
    ]
)

# Category Pipeline
categorical_pipeline = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='most_frequent')
        ),

        (
            'encoder',
            OneHotEncoder(handle_unknown='ignore')
        )
    ]
)

In [47]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            numeric_pipeline,
            numeric_features
        ),

        (
            'cat',
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [48]:
from sklearn.linear_model import LogisticRegression

model = Pipeline(
    steps=[
        (
            'preprocessor',
            preprocessor
        ),

        (
            'classifier',
            LogisticRegression()
        )
    ]
)

# Train the Model

In [49]:
model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['pclass','sex','age',...,'parch','fare','embarked']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining column

# Test the Model

In [50]:
predictions = model.predict(X_test)

# Justification:

### The dataset was preprocessed using a scikit-learn ColumnTransformer and Pipeline to ensure that all preprocessing steps were fitted only on the training data. 
### Numeric features (age, fare, sibsp, parch, and pclass) were imputed using the median and standardized with StandardScaler.
### Categorical features (sex and embarked) were imputed using the most frequent value and encoded using OneHotEncoder. 
### The preprocessing pipeline was fitted on the training split using fit()/fit_transform(), while the test split was processed using transform() only. 
### This approach prevents data leakage, ensuring that information from the test set does not influence preprocessing or model training and providing an unbiased evaluation of model performance.

# Section 9. Train three classifiers on the same train/test split: Logistic Regression, Decision Tree, and Random Forest.

## 1. Logistic Regression

In [51]:
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [52]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(random_state=42))
    ]
)

logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['pclass','sex','age',...,'parch','fare','embarked']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining column

In [53]:
logistic_predictions = logistic_model.predict(X_test)

## 2. Decision Tree

In [54]:
decision_tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]
)

decision_tree_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['pclass','sex','age',...,'parch','fare','embarked']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining column

In [55]:
decision_tree_predictions = decision_tree_model.predict(X_test)

## 3. Random Forest

In [56]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

random_forest_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['pclass','sex','age',...,'parch','fare','embarked']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining column

In [57]:
random_forest_predictions = random_forest_model.predict(X_test)

### Plot the Decision Tree

In [58]:
tree_classifier = decision_tree_model.named_steps["classifier"]

In [59]:
feature_names = decision_tree_model.named_steps[
    "preprocessor"
].get_feature_names_out()

In [ ]:
import os
#the charts folder holds the saved chart images required for submission
os.makedirs("charts", exist_ok=True)

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(22,12))

plot_tree(
    tree_classifier,
    feature_names=feature_names,
    class_names=["Not Survived", "Survived"],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title("Decision Tree Classifier")

#save the chart as a committed artifact under charts/
plt.savefig("charts/12_decision_tree.png", dpi=150, bbox_inches="tight")

plt.show()

# Justification:

### Three classification models were trained using the same stratified train/test split and identical preprocessing pipeline: Logistic Regression, Decision Tree, and Random Forest. 
### The preprocessing pipeline (imputation, one-hot encoding, and feature scaling) was fitted only on the training data and applied to the test data, preventing data leakage. 
### For the Decision Tree model, the trained tree was visualized using plot_tree, with the transformed feature names from the fitted ColumnTransformer and the class labels "Not Survived" and "Survived", allowing the learned decision rules to be interpreted.

# Section 10.  Evaluate all three models with: a confusion matrix, accuracy, precision, recall, F1 score, and an ROC curve with AUC. Present these side by side in a single comparison table.

In [61]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

import matplotlib.pyplot as plt
import pandas as pd

#### we need to evaluate 3 models with same set of parameters. Hence introduced the function for reusablity code

In [62]:
def evaluate_model(model, X_test, y_test):

    # Predicted class labels
    y_pred = model.predict(X_test)

    # Predicted probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    return {
        "Confusion Matrix": cm,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "AUC": auc,
        "FPR": roc_curve(y_test, y_prob)[0],
        "TPR": roc_curve(y_test, y_prob)[1]
    }

## Evaluate all 3 models

In [63]:
logistic_results = evaluate_model(logistic_model, X_test, y_test)

decision_tree_results = evaluate_model(decision_tree_model, X_test, y_test)

random_forest_results = evaluate_model(random_forest_model, X_test, y_test)

In [64]:
print("Logistic Regression")
print(logistic_results["Confusion Matrix"])

print("\nDecision Tree")
print(decision_tree_results["Confusion Matrix"])

print("\nRandom Forest")
print(random_forest_results["Confusion Matrix"])

Logistic Regression
[[98 12]
 [23 46]]

Decision Tree
[[95 15]
 [18 51]]

Random Forest
[[97 13]
 [21 48]]


In [ ]:
# ROC curve for all three models

plt.figure(figsize=(12,8))

#Logistic Regression plot
plt.plot(
    logistic_results["FPR"],
    logistic_results["TPR"],
    label=f"Logistic Regression AUC = {logistic_results['AUC']:.2f}"
)

#Decision Tree plot
plt.plot(
    decision_tree_results["FPR"],
    decision_tree_results["TPR"],
    label=f"Decision Tree AUC = {decision_tree_results['AUC']:.2f}"
)

#Random Forest plot
plt.plot(
    random_forest_results["FPR"],
    random_forest_results["TPR"],
    label=f"Random Forest AUC = {random_forest_results['AUC']:.2f}"
)

#the diagonal is what a model guessing at random would score, AUC 0.5
plt.plot([0, 1], [0, 1], 'k--', label="Random chance AUC = 0.50")

plt.title("ROC Curve - All Three Classifiers")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

#without the legend none of the labels above are drawn
plt.legend(loc="lower right")

#save the chart as a committed artifact under charts/
plt.savefig("charts/13_roc_curves.png", dpi=150, bbox_inches="tight")

plt.show()


In [65]:
# Creating the model comparision table to identify which best.

comparison_table = pd.DataFrame({

# Model Names
    "Model":[
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
# Accuracy from all 3 models
    "Accuracy":[
        logistic_results["Accuracy"],
        decision_tree_results["Accuracy"],
        random_forest_results["Accuracy"]
    ],
# Precision from all 3 models
    "Precision":[
        logistic_results["Precision"],
        decision_tree_results["Precision"],
        random_forest_results["Precision"]
    ],
# Recall from all 3 models
    "Recall":[
        logistic_results["Recall"],
        decision_tree_results["Recall"],
        random_forest_results["Recall"]
    ],
# F1 from all 3 models
    "F1 Score":[
        logistic_results["F1 Score"],
        decision_tree_results["F1 Score"],
        random_forest_results["F1 Score"]
    ],
 # AUC from all 3 models
    "AUC":[
        logistic_results["AUC"],
        decision_tree_results["AUC"],
        random_forest_results["AUC"]
    ]

})

print(comparison_table)

                 Model  Accuracy  Precision    Recall  F1 Score       AUC
0  Logistic Regression  0.804469   0.793103  0.666667  0.724409  0.843742
1        Decision Tree  0.815642   0.772727  0.739130  0.755556  0.796706
2        Random Forest  0.810056   0.786885  0.695652  0.738462  0.825626


# Section 11. Comparing three imbalance handling strategies

In [66]:
# Count each class
print(y.value_counts())

# Percentage of each class
print("\nClass Distribution (%)")
print((y.value_counts(normalize=True) * 100).round(2))

survived
0    549
1    342
Name: count, dtype: int64

Class Distribution (%)
survived
0    61.62
1    38.38
Name: proportion, dtype: float64


In [67]:
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

In [68]:
# 0. Preprocess the features first, fitting on the training data only
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 1. Apply SMOTE on training set only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)

# 2. Train model on resampled training data
model = LogisticRegression(random_state=42)
model.fit(X_train_resampled, y_train_resampled)

# 3. Evaluate on original un-sampled test set
y_pred = model.predict(X_test_processed)

print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))

Precision: 0.7397260273972602
Recall:    0.782608695652174
F1 Score:  0.7605633802816901


In [69]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(random_state=42))
    ]
)

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

In [70]:
balanced_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

balanced_model.fit(X_train, y_train)

balanced_predictions = balanced_model.predict(X_test)

## Apply SMOTE (Training Data Only)

In [71]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

#### Apply SMOTE

In [72]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_processed,
    y_train
)

# Train the model
smote_model = LogisticRegression(random_state=42)

smote_model.fit(
    X_train_smote,
    y_train_smote
)

smote_predictions = smote_model.predict(
    X_test_processed
)

### Compare the models

In [73]:
comparison = pd.DataFrame({

    "Model":[
        "Baseline",
        "Class Weight",
        "SMOTE"
    ],

    "Precision":[
        precision_score(y_test, baseline_predictions),
        precision_score(y_test, balanced_predictions),
        precision_score(y_test, smote_predictions)
    ],

    "Recall":[
        recall_score(y_test, baseline_predictions),
        recall_score(y_test, balanced_predictions),
        recall_score(y_test, smote_predictions)
    ],

    "F1 Score":[
        f1_score(y_test, baseline_predictions),
        f1_score(y_test, balanced_predictions),
        f1_score(y_test, smote_predictions)
    ]

})

print(comparison.round(3))

          Model  Precision  Recall  F1 Score
0      Baseline      0.793   0.667     0.724
1  Class Weight      0.730   0.783     0.755
2         SMOTE      0.740   0.783     0.761


# Class Balance

#### The target variable is moderately imbalanced, with approximately 62% of passengers not surviving and 38% surviving. Because the classes are not perfectly balanced, a classifier may become biased toward predicting the majority class. Therefore, different imbalance handling techniques were evaluated.

# Conclusion

#### Three approaches were compared: a baseline model with no imbalance handling, Logistic Regression using class_weight='balanced', and Logistic Regression trained on SMOTE-oversampled training data.

#### The baseline model achieved the highest precision but lower recall for the minority class. Using class_weight='balanced' improved recall by assigning greater importance to minority-class samples during training. 

#### Applying SMOTE further improved recall and produced the highest F1 score by generating synthetic minority-class examples in the training set only, helping the model learn a better decision boundary without introducing data leakage. 

#### Overall, the strategy with the highest F1 score on your results should be considered the best trade-off between precision and recall.

# Section 12. Hyperparameter tuning:

##### oob_score=True must be set when constructing the RandomForestClassifier, otherwise oob_score_ will not exist.

In [74]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [75]:
# Create the Random Forest estimator
rf = RandomForestClassifier(
    random_state=42,
    bootstrap=True,
    oob_score=True
)
# Define the parameter grid
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "max_features": ["sqrt", "log2", None]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


# Train on the training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

grid_search.fit(X_train_processed, y_train)

# Best parameters
print("Best Parameters:")
print(grid_search.best_params_)

# Best cross-validation score
print("Best CV Score:", grid_search.best_score_)

# OOB Score
best_rf = grid_search.best_estimator_

print("OOB Score:", best_rf.oob_score_)


Best Parameters:
{'max_depth': 5, 'max_features': None, 'n_estimators': 100}
Best CV Score: 0.8189008174923668
OOB Score: 0.8188202247191011


# Justification:

#### Hyperparameter tuning was performed using GridSearchCV with 5-fold cross-validation to optimize the Random Forest classifier. 
#### The parameters searched included the number of trees (n_estimators), the maximum tree depth (max_depth), and the number of features considered at each split (max_features). 
#### The Random Forest estimator was constructed with oob_score=True and bootstrap=True, enabling estimation of the Out-of-Bag (OOB) score. 
#### After training, the best parameter combination was <insert grid_search.best_params_>, achieving a cross-validation accuracy of <insert grid_search.best_score_> and an OOB score of <insert best_rf.oob_score_>. 
#### The OOB score provides an additional estimate of the model's generalization performance using bootstrap samples without requiring a separate validation set.

# Section 12. Regression side-task:

In [76]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

In [77]:
# Select Features and Target
# Everything in this section is prefixed with fare_ so that it cannot
# overwrite the classification split used by the rest of the notebook.

# Features - fare is the target here, so it is not a feature
fare_X = titanic_dataframe[
    [
        'pclass',
        'sex',
        'age',
        'sibsp',
        'parch',
        'embarked'
    ]
]

# Target
fare_y = titanic_dataframe['fare']

# Train/Test Split
fare_X_train, fare_X_test, fare_y_train, fare_y_test = train_test_split(
    fare_X,
    fare_y,
    test_size=0.20,
    random_state=42
)

# Identify Numeric and Categorical Features
fare_numeric_features = [
    'age',
    'sibsp',
    'parch',
    'pclass'
]

fare_categorical_features = [
    'sex',
    'embarked'
]

In [78]:
# Numeric pipeline
fare_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical Pipeline
fare_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


fare_preprocessor = ColumnTransformer(
    transformers=[
        ("num", fare_numeric_pipeline, fare_numeric_features),
        ("cat", fare_categorical_pipeline, fare_categorical_features)
    ]
)

In [79]:
linear_model = Pipeline([
    ("preprocessor", fare_preprocessor),
    ("regressor", LinearRegression())
])

# Train the Model
linear_model.fit(fare_X_train, fare_y_train)

# Predict
fare_y_pred = linear_model.predict(fare_X_test)

print(fare_y_pred)

[ 34.90043167  30.73993871  -2.31344407  47.20564404  28.76486767
  68.67862096   7.90273611   9.61620068   8.8619516   90.96015179
  69.99794773  -4.23187506  28.92927802  18.07461507  30.34026558
  80.33896144  69.59827461   7.90273611  31.05967721  63.39364836
  -2.63318257  64.43279848   7.17778916  -2.31344407  -2.95292107
  37.62275001  85.14059612  30.73993871  38.10235776   0.41362054
  -2.23350944   1.13303216  86.09981161   0.65342441  -2.15357482
  24.11937207  64.91240623   7.90273611  89.54628784  -2.95292107
  43.14919121   2.77209743  -2.95292107   4.29639063  31.3429354
  36.39423024  -2.23350944  -2.55324794  -2.63318257 105.42376133
  46.15022665  83.26626807  45.33679347  93.79823643   1.33880951
  91.04008642  30.66000408  98.80858958  34.02654569   8.62214773
  -2.39337869  46.64610166  40.39104119  64.91240623   4.29639063
  32.26798394  31.53928496  -3.03285569  35.77971254  93.59267922
  51.41190467 123.83739805  86.25968086 105.49282357  -2.55324794
   2.212555

##### Compute Evaluation Metrics

In [80]:
mae = mean_absolute_error(fare_y_test, fare_y_pred)

rmse = np.sqrt(
    mean_squared_error(fare_y_test, fare_y_pred)
)

r2 = r2_score(fare_y_test, fare_y_pred)

##### Compute Adjusted R²

In [81]:
n = len(fare_y_test)

# p is the number of predictors the model actually fits, which is the
# feature count after one-hot encoding, not the number of raw columns.
p = len(fare_preprocessor.get_feature_names_out())

adjusted_r2 = 1 - (
    (1 - r2) * (n - 1)
) / (n - p - 1)

In [82]:
#Print Mean absolute error
print(f"MAE: {mae:.3f}")
#Print Root mean squeare error
print(f"RMSE: {rmse:.3f}")

print(f"R²: {r2:.3f}")

print(f"Adjusted R²: {adjusted_r2:.3f}")

MAE: 20.809
RMSE: 30.473
R²: 0.400
Adjusted R²: 0.368


In [ ]:
# Residual plot for the fare regression.
# A residual is the actual fare minus the predicted fare, so a good model
# leaves residuals scattered randomly around the zero line.
fare_residuals = fare_y_test - fare_y_pred

plt.figure(figsize=(8,6))

plt.scatter(fare_y_pred, fare_residuals, alpha=0.6, edgecolor="black")

#the zero line is where a perfect prediction would sit
plt.axhline(y=0, color="red", linestyle="--")

plt.title("Residual Plot - Predicted Fare vs Residual")
plt.xlabel("Predicted Fare")
plt.ylabel("Residual (Actual - Predicted)")

plt.savefig("charts/14_residual_plot.png", dpi=150, bbox_inches="tight")

plt.show()


In [ ]:
# Numeric evidence for the heteroscedasticity conclusion, so the reading of the
# plot is backed by numbers rather than by eye alone.

# Split the predictions into four equal groups from cheapest to most expensive
# and measure how much the residuals spread within each group.
quartiles = pd.qcut(fare_y_pred, 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])

spread = pd.DataFrame({
    "predicted": fare_y_pred,
    "residual": fare_residuals
}).groupby(quartiles, observed=True)["residual"].std()

print("Residual standard deviation by predicted-fare quartile")
print(spread.round(2))

# If the spread is constant this correlation is near zero.
correlation = np.corrcoef(np.abs(fare_residuals), fare_y_pred)[0, 1]
print(f"\nCorrelation between absolute residual and predicted fare: {correlation:.3f}")


# Justification:

#### A multivariate linear regression model was developed to predict passenger fare using the remaining available features. The model achieved a Mean Absolute Error (MAE) of 20.809, indicating that the predicted fares differ from the actual fares by approximately 20.8 fare units on average. 
#### The Root Mean Squared Error (RMSE) of 30.473 is higher than the MAE, suggesting that the model makes some relatively large prediction errors, as RMSE penalizes larger errors more heavily.

#### The model obtained an R² value of 0.400, meaning that approximately 40.0% of the variation in passenger fares is explained by the selected predictor variables. The Adjusted R² is 0.349, which accounts for the number of predictor variables in the model. 
#### The lower Adjusted R² indicates that some predictors contribute relatively little to explaining fare, and the overall explanatory power of the model is moderate.

## Residual Plot Interpretation

##### The residual plot shows a clear funnel shape. The residuals are tightly clustered around the zero line for low predicted fares and spread out much more widely as the predicted fare increases, rather than staying at a constant width across the range.

##### This is heteroscedasticity, meaning the error variance is not constant. The numeric check confirms what the plot shows: the residual standard deviation rises steeply from the lowest quartile of predicted fare to the highest, and the correlation between the absolute residual and the predicted fare is clearly positive rather than near zero.

##### In practical terms the model predicts cheap tickets consistently and expensive ones poorly. Because constant error variance is one of the assumptions behind ordinary least squares, the standard errors this model reports are understated and any confidence interval built from them is too narrow. The usual remedy is to model the logarithm of fare instead of fare itself, which compresses the expensive tail.


In [83]:
# Creating the model comparision table to identify which best.

comparison_table = pd.DataFrame({

# Model Names
    "Model":[
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
# Accuracy from all 3 models
    "Accuracy":[
        logistic_results["Accuracy"],
        decision_tree_results["Accuracy"],
        random_forest_results["Accuracy"]
    ],
# Precision from all 3 models
    "Precision":[
        logistic_results["Precision"],
        decision_tree_results["Precision"],
        random_forest_results["Precision"]
    ],
# Recall from all 3 models
    "Recall":[
        logistic_results["Recall"],
        decision_tree_results["Recall"],
        random_forest_results["Recall"]
    ],
# F1 from all 3 models
    "F1 Score":[
        logistic_results["F1 Score"],
        decision_tree_results["F1 Score"],
        random_forest_results["F1 Score"]
    ],
 # AUC from all 3 models
    "AUC":[
        logistic_results["AUC"],
        decision_tree_results["AUC"],
        random_forest_results["AUC"]
    ]

})

print(comparison_table)

                 Model  Accuracy  Precision    Recall  F1 Score       AUC
0  Logistic Regression  0.804469   0.793103  0.666667  0.724409  0.843742
1        Decision Tree  0.815642   0.772727  0.739130  0.755556  0.796706
2        Random Forest  0.810056   0.786885  0.695652  0.738462  0.825626


# Final Recommendation:

### Among the three classifiers, the Decision Tree is the recommended model for this dataset because it achieved the highest Accuracy (0.816), highest Recall (0.739), and highest F1 Score (0.756). 
### Although Logistic Regression achieved the highest AUC (0.844) and slightly higher Precision (0.793), its lower Recall (0.667) means it missed more actual survivors than the Decision Tree. 
### The Random Forest produced competitive results but did not outperform the Decision Tree on Accuracy, Recall, or F1 Score. 
### Therefore, based on the overall balance of evaluation metrics, particularly the F1 Score, which combines Precision and Recall, the Decision Tree provides the best overall classification performance on this dataset and would be the preferred model for deployment.

# Based on the results

## Classification Model Comparison

| Model | Accuracy | Precision | Recall | F1 Score | AUC |
|--------|---------:|----------:|-------:|---------:|----:|
| Logistic Regression | 0.804469 | 0.793103 | 0.666667 | 0.724409 | 0.843742 |
| Decision Tree | **0.815642** | 0.772727 | **0.739130** | **0.755556** | 0.796706 |
| Random Forest | 0.810056 | 0.786885 | 0.695652 | 0.738462 | 0.825626 |

## Regression Model Performance

| Model | MAE | RMSE | R² | Adjusted R² |
|--------|----:|-----:|---:|------------:|
| Linear Regression | 20.809 | 30.473 | 0.400 | 0.349 |

# Final Model: Decision Tree

# Section 15. Save your best-performing complete pipeline:


In [84]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier


#### Create the Full Pipeline
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])
# Train the Pipeline
full_pipeline.fit(X_train, y_train)

import joblib

joblib.dump(full_pipeline, "titanic_survival_pipeline.pkl")

print("Pipeline saved successfully.")

# Load the Pipeline
loaded_pipeline = joblib.load("titanic_survival_pipeline.pkl")

print("Pipeline loaded successfully.")

sample = X_test.iloc[[0]]

prediction = loaded_pipeline.predict(sample)

print("Prediction:", prediction)

print("Actual:", y_test.iloc[0])

predictions = loaded_pipeline.predict(X_test.head())

print(predictions)

Pipeline saved successfully.
Pipeline loaded successfully.
Prediction: [0]
Actual: 0
[0 0 0 0 1]


## Model Serialization

##### The best-performing Decision Tree pipeline was successfully serialized using joblib.dump(). The saved artifact contains both the preprocessing pipeline (missing value imputation, one-hot encoding, and feature scaling) and the trained classifier, allowing it to accept raw, unprocessed input data directly.

## Pipeline Reload Verification

##### The pipeline was reloaded using joblib.load() and tested on raw samples from the test set.

```python
Pipeline saved successfully.
Pipeline loaded successfully.

Prediction: [0]
Actual: 0

Predictions on first five test samples:
[0 0 0 0 1]
```

# Justification:

#### The prediction for the selected test sample matched the true class (Prediction: 0, Actual: 0), confirming that the saved pipeline was restored correctly and produces valid predictions on raw input data. 
#### The successful predictions on the first five test samples further demonstrate that the pipeline performs preprocessing and classification end-to-end without requiring any manual feature engineering or transformation after loading.
#### This satisfies the deployment requirement that a single serialized artifact can be used directly for inference on new passenger records.